# pytorch3d Wheel Builder

Run this notebook ONCE on Kaggle with T4 GPU + Internet enabled.

It compiles pytorch3d for sm_75 (T4) and saves the `.whl` file to `/kaggle/working/`.

**After it finishes:**
1. Go to **Output** tab → download `pytorch3d-*.whl`
2. Go to **Kaggle → Datasets → New Dataset**
3. Upload the `.whl` file, name the dataset: `pytorch3d-t4-whl`
4. Add this dataset to the `forma-hood-validation` notebook under **Data** tab

Every future validation run will install from the wheel in ~10 seconds instead of ~10 minutes.

In [ ]:
import os, subprocess
from pathlib import Path

# Verify T4 GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())

# Build pytorch3d wheel for sm_75 (T4) only
os.environ['FORCE_CUDA'] = '1'
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'

wheel_dir = Path('/kaggle/working/wheels')
wheel_dir.mkdir(exist_ok=True)

print('Building pytorch3d wheel for sm_75...')
subprocess.run(
    ['pip', 'wheel',
     'git+https://github.com/facebookresearch/pytorch3d.git@stable',
     '--wheel-dir', str(wheel_dir),
     '--no-deps'],
    check=True
)

wheels = list(wheel_dir.glob('*.whl'))
print('\nBuilt wheels:')
for w in wheels:
    size_mb = w.stat().st_size / 1024 / 1024
    print(f'  {w.name}  ({size_mb:.1f} MB)')

print('\nDone. Download the .whl from the Output tab, then upload to a Kaggle dataset named pytorch3d-t4-whl')